In [4]:
import os
from dotenv import load_dotenv
from typing import List, Literal, Optional, TypedDict, Dict
from langgraph.graph import END, StateGraph, state
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
import pandas as pd
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

In [5]:
load_dotenv()
envVars = pd.Series(os.environ)
envVars = envVars[["OPENAI_API_KEY", "AGENT_MODEL", "AGENT_MAX_RETRIES"]]
for i in envVars.index: print(i, ": ", envVars.loc[i][:16], sep = "")

OPENAI_API_KEY: sk-proj-sRDo3IE5
AGENT_MODEL: gpt-5.4-mini-202
AGENT_MAX_RETRIES: 3


In [6]:
set_llm_cache(InMemoryCache())

In [7]:
def _llm() -> ChatOpenAI:
    model = os.environ.get("AGENT_MODEL", "gpt-4o-mini")
    # Low temperature — we want reproducible, focused code.
    return ChatOpenAI(model = model, temperature = 0.2)

llm = _llm()

In [8]:
task = {
    "name": "column_maximim_strategy",
    "problem": "Given a numpy matrix A, find a probability vector as another column numpy matrix p, such that there exist no probability vector q satisfies min(A@q) > min(A@p)",
    "signature": "def column_maximin(A: numpy.matrix) -> numpy.matrix:",
    "public_tests": [
      "assert bool(min(np.matrix([[-54, 32],[62, -37],[-13, 7]])@column_maximin(np.matrix([[-54, 32],[62, -37],[-13, 7]])))[0, 0] >= -0.395)"
    ],
    "hidden_tests": [
      "assert bool(min(np.matrix([[-65, 39],[62, 40],[-3, -11]])@column_maximin(np.matrix([[-65, 39],[62, 40],[-3, -11]])))[0, 0] >= -7.43)"
    ]
}

In [9]:
class TestResult(TypedDict):
    """Outcome of running the generated code against the task's tests."""
    passed: bool
    stdout: str
    stderr: str
    failing_test: Optional[str]

Workflow = Literal["Init", "Specify", "Generate", "Test", "Repair", "Done", "Fail", "Plan"]

class AgentState(TypedDict, total=False):
    # --- task description (immutable after Init) -------------------------
    problem: str            # natural-language problem statement
    signature: str          # required function signature, e.g. "def foo(x):"
    public_tests: List[str] # python `assert` statements
    hidden_tests: List[str] # extra tests only used to grade final solution

    # --- mutable during the run ----------------------------------------
    workflow: Workflow      # current control-flow state  (TLA+: pc)
    code: str               # latest generated solution    (TLA+: code)
    last_result: Optional[TestResult]   # last Test outcome (TLA+: tested)
    retries: int            # number of repair attempts so far (TLA+: retries)
    max_retries: int        # upper bound on retries           (TLA+: MaxRetries)

    approach: str
    subtasks: List[str]
    plusCalSpec: str
    unprovedSubtasks: List[str]
    provedSubtasks: Dict[str, str]

    # --- auditing ------------------------------------------------------
    history: List[str]      # human-readable trace for the report

In [10]:
currentState: AgentState = {
    "problem": task["problem"],
    "signature": task["signature"],
    "public_tests": task["public_tests"],
    "hidden_tests": task.get("hidden_tests", []),
    "workflow": "Init",
    "code": "",
    "plusCalSpec": "",
    "last_result": None,
    "retries": 0,
    "max_retries": 4,
    "history": [],
    "approach": "",
    "subtasks": [],
    "unprovedSubtasks": [],
    "provedSubtasks": {},
}

In [11]:
def init_action(state: AgentState) -> AgentState:
    """Init: set the workflow counters. Mirrors TLA+ ``Init``."""
    return {
        "workflow": "Plan",
        "retries": 0,
        "max_retries": state.get("max_retries", int(os.environ.get("AGENT_MAX_RETRIES", "3"))),
        "history": state.get("history", []) + ["Init -> Plan"],
    }

In [12]:
planSysMessage = (
    "You are a careful programmer and has been given a problem."
    "First analyze the problem to see if it is equivalent to problems with available solutions"
    "If yes then summarize a solution into a plan"
    "If no then suggest a plan"
    "Note that at this stage we are looking for a plan, not an implementation"
    "Try to describe the plan using natural language and PlusCal"
)
def plan_action(state: AgentState) -> AgentState:
    prompt = (
        f"Problem:\n{state['problem']}\n\n"
        f"Required signature:\n{state['signature']}\n\n"
        "Before generating the implementation, try to design an approach to solve this problem"
    )
    resp = llm.invoke([ SystemMessage(content = planSysMessage), 
                        HumanMessage(content = prompt)])
    return {"approach": resp.content, 
            "workflow": "Spec"}

In [21]:
splitSysMessage = (
    "You are a careful programmer and has a plan."
    "Split the plan into sub-tasks, "
    "such that each task focus on one component of the plan."
    "If the task is easy then one sub-task is accepable."
    "We need your response to be arranged such that each sub-tasks start with ' --- '"
    "and does not have this prefix anywhere else"
    "Note that at this stage we wish to specify the procedure, instead of implementing"
    "Breakdown the steps and specify them using PlusCal"
)
def split_action(state: AgentState) -> AgentState:
    prompt = (
        f"Problem:\n{state['problem']}\n\n"
        f"Required signature:\n{state['signature']}\n\n"
        f"Approach:\n{state['approach']}"
    )
    resp = llm.invoke([ SystemMessage(content = splitSysMessage), 
                        HumanMessage(content = prompt)])
    subtasks = resp.content.split(" --- ")
    subtasks.remove("")
    return {"subtasks": subtasks, 
            "unprovedSubtasks": subtasks.copy(),
            "provedSubtasks": {},
            "workflow": "Spec"}

In [22]:
proveSysMessage = (
    "You are a formal verification expert. "
    "Your job is to take a PlusCal sub-task and write the TLA+ correctness proofs (TypeOK, Invariants, or Theorems) for it. "
    "Output the original PlusCal specification along with your added proofs."
)

def prove_action(state: AgentState) -> AgentState:
    unproved = state.get("unprovedSubtasks", []).copy()
    proved = state.get("provedSubtasks", {}).copy()
    print(len(unproved))
    current_task = unproved.pop(0)
    # Prove one for each time
    prompt = f"Problem context:\n{state['approach']}\n\nCurrent Sub-task to prove:\n{current_task}"
    resp = llm.invoke([SystemMessage(content = proveSysMessage), HumanMessage(content = prompt)])
    proved[current_task] = resp.content
    return {
        "unprovedSubtasks": unproved,
        "provedSubtasks": proved
    }

def route_after_prove(state: AgentState) -> str:
    unproved = state.get("unprovedSubtasks", [])
    if len(unproved) > 0: return "continue_proving"
    else: return "all_proved"

In [23]:
def newPlanAgent() -> state.CompiledStateGraph:
    graph = StateGraph(AgentState)
    graph.add_node("init", init_action)
    graph.add_node("plan", plan_action)
    graph.add_node("split", split_action)
    graph.add_node("prove", prove_action)

    graph.set_entry_point("init")
    graph.add_edge("init", "plan")
    graph.add_edge("plan", "split")
    graph.add_edge("split", "prove")

    graph.add_conditional_edges(
        "prove",
        route_after_prove,
        {"continue_proving": "prove",
         "all_proved": END}
    )

    return graph.compile()

agent = newPlanAgent()
plannedState:AgentState = agent.invoke(currentState, config = {"recursion_limit": 16})

# for k in plannedState.keys(): print(k, ": ", plannedState[k], sep = "")

8
7
6
5
4
3
2
1


In [24]:
# print(plannedState["approach"])
for k in plannedState["provedSubtasks"].keys():
    print(k, ": ", plannedState["provedSubtasks"][k], sep = "", end = "\n\n\n\n")

Normalize the input and identify dimensions
```
procedure ColumnMaximin(A)
    define m := NumberOfRows(A)
    define n := NumberOfColumns(A)
```

: Below is the original PlusCal sub-task with TLA+ proofs added for the normalization and dimension-identification step.

```tla
---- MODULE ColumnMaximin ----
EXTENDS Naturals, Sequences, RealNumbers

(***************************************************************************)
(* Original PlusCal-style specification fragment                           *)
(***************************************************************************)

(* procedure ColumnMaximin(A)
    define m := NumberOfRows(A)
    define n := NumberOfColumns(A)
*)

(***************************************************************************)
(* TLA+ proof obligations for this sub-task                                *)
(***************************************************************************)

CONSTANT A

VARIABLES m, n

(*
  The sub-task is purely definitional:
  - m is t

In [25]:
print(plannedState["provedSubtasks"].keys())

dict_keys(['Normalize the input and identify dimensions\n```\nprocedure ColumnMaximin(A)\n    define m := NumberOfRows(A)\n    define n := NumberOfColumns(A)\n```\n\n', 'Introduce the LP decision variables\n```\n    define p[1..n]   // probability vector over columns\n    define v         // guaranteed minimum value\n```\n\n', 'State the optimization goal\n```\n    // Maximize v such that A * p is componentwise at least v,\n    // while p remains a probability vector\n```\n\n', 'Encode the feasibility and simplex constraints\n```\n    require for all j in 1..n: p[j] >= 0\n    require Sum(j in 1..n, p[j]) = 1\n    require for all i in 1..m:\n        Sum(j in 1..n, A[i,j] * p[j]) >= v\n```\n\n', 'Convert the maximin problem into a linear program\n```\n    // Equivalent LP:\n    // maximize v\n    // subject to A*p >= v*1, p >= 0, sum(p)=1\n    // This is a standard linear program\n```\n\n', 'Specify the solver-oriented form\n```\n    // If using a minimization LP solver:\n    // minimize